# M4 · Analyse the traces: can caching help at all?

A cache only helps if some experts are used **much more often** than others
("hot" experts), or if tokens **re-use** the experts of recent tokens.
If every expert were used equally and randomly, no cache policy could beat random.

**Kaggle:** Accelerator = **None** (CPU is enough, saves GPU quota), Internet = On.
**Inputs:** Add Input → Datasets → **moe-traces** (from M2).

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import config
from src.files import load_trace

traces = {name: load_trace(name) for name in ["wikitext", "code"]}
for name, tr in traces.items():
    print(f"{name:9s} trace shape {tr.shape}  ->  {tr.shape[0]:,} tokens, {tr.size:,} expert requests")

## 1. How often is each expert used?
`counts[layer, expert]` = number of tokens that used that expert in that layer.

In [ ]:
def expert_usage(trace, num_experts=64):
    tokens, layers, k = trace.shape
    counts = np.zeros((layers, num_experts), dtype=np.int64)
    for l in range(layers):
        counts[l] = np.bincount(trace[:, l].ravel(), minlength=num_experts)
    return counts

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, tr) in zip(axes, traces.items()):
    counts = expert_usage(tr)
    share = counts / counts.sum(axis=1, keepdims=True)     # fraction of requests per expert
    share = -np.sort(-share, axis=1)                        # sort hottest -> coldest in each layer
    im = ax.imshow(share, aspect="auto", cmap="viridis")
    ax.set(title=f"{name}: usage share (experts sorted hot -> cold)",
           xlabel="expert rank (0 = most used)", ylabel="layer")
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, "M4_usage.png"), dpi=130); plt.show()

If all experts were used equally, every cell would be 1/64 ≈ 0.016 and the picture
would be flat. A bright left side means a few experts get most of the work.

## 2. Three numbers per layer
- **Hot-16 share**: fraction of requests served by the 16 most-used experts.
  This is the hit rate of a *fixed* cache holding those 16 (equal use would give 16/64 = 0.25).
- **Reuse rate**: fraction of token t's experts that token t-1 also used
  (random routing would give 8/64 = 0.125).
- **Entropy** (bits): how spread out the usage is. Equal use of 64 experts = 6 bits; lower = more skewed.

In [ ]:
def hot_share(trace, n_hot=16):
    counts = expert_usage(trace)
    top = np.sort(counts, axis=1)[:, ::-1][:, :n_hot].sum(axis=1)    # requests to the n_hot hottest
    return top / counts.sum(axis=1)

def reuse_rate(trace):
    tokens, layers, k = trace.shape
    rates = []
    for l in range(layers):
        shared = 0
        for now, before in zip(trace[1:, l], trace[:-1, l]):
            shared += len(set(now) & set(before))                     # experts used by both tokens
        rates.append(shared / ((tokens - 1) * k))
    return np.array(rates)

def entropy_bits(trace):
    counts = expert_usage(trace).astype(float)
    p = counts / counts.sum(axis=1, keepdims=True)
    return -(p * np.log2(np.where(p > 0, p, 1))).sum(axis=1)          # 0 * log(0) counts as 0

stats = {}
for name, tr in traces.items():
    stats[name] = {"hot16": hot_share(tr), "reuse": reuse_rate(tr[:50_000]),   # 50k tokens is plenty
                   "entropy": entropy_bits(tr)}
    print(f"{name:9s} mean hot-16 share {stats[name]['hot16'].mean():.2f} | "
          f"mean reuse {stats[name]['reuse'].mean():.2f} | mean entropy {stats[name]['entropy'].mean():.2f} bits")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
baselines = {"hot16": 16/64, "reuse": 8/64, "entropy": 6.0}
titles = {"hot16": "Share served by 16 hottest experts", "reuse": "Reuse from previous token",
          "entropy": "Usage entropy (bits)"}
for ax, key in zip(axes, ["hot16", "reuse", "entropy"]):
    for name in traces:
        ax.plot(stats[name][key], marker="o", label=name)
    ax.axhline(baselines[key], ls="--", c="gray", label="if routing were random")
    ax.set(title=titles[key], xlabel="layer"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, "M4_layer_stats.png"), dpi=130); plt.show()

## What to write in `docs/M4.md`
- Are the lines above the "random" dashed lines? By how much? → caching can help.
- Which layers are most skewed? (Those need fewer cache slots → M7.)
- Is code different from English?

## ✅ Done when
Both plots are saved and you can answer "can caching help?" with numbers.